## Step 1: Install Dependencies

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio"

## Step 2: Set GOOGLE_API_KEY in Colab

In [ ]:
# To use Gemini models, you need to set the GOOGLE_API_KEY environment variable in Colab.
# You can store your API key in Colab's 'Secrets' tab (🔑 icon on the left panel)
# and then access it like this:
from google.colab import userdata
import os

# Set your API key from Colab secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## Step 3: Confirm Node/NPM availability

In [ ]:
# Many MCP servers are distributed as Node packages runnable via npx.
!node --version
!npx --version

v20.19.0
10.8.2


If the above command shows an error, run the following cell to install Node.js and npm.

In [ ]:
# If missing:
# !apt-get -qq update
# !apt-get -qq install -y nodejs npm
# !node --version
# !npx --version

# Uncomment and run the above lines if node or npx are not found.

## Step 4: Choose third-party MCP servers

The exercise requires using at least two third-party MCP servers. You can find a list of official MCP servers [here](https://modelcontext.io/docs/servers/official-servers/).

For this exercise, we will proceed with the following MCP servers:
1.  **Filesystem Server**: `@modelcontextprotocol/server-filesystem`
2.  **Git Server**: `mcp-server-git`

Let's prepare the environment for these servers.

## Step 5: Launch MCP servers in Colab (stdio transport)

In MCP, your agent typically communicates with servers via stdio (subprocess). In Colab, that means the agent spawns subprocesses that run `npx -y <server-package> ...` or `python -m <server-module> ...`.

First, we'll define a working directory that both the filesystem and git servers will operate within. This directory will be created if it doesn't already exist.

In [ ]:
import os
from pathlib import Path

# Define a working directory for the MCP servers
WORKDIR = Path("/content/mcp_workspace")
WORKDIR.mkdir(exist_ok=True, parents=True)
print(f"Created MCP workspace directory: {WORKDIR}")

# Ensure git is installed for the git server
!apt-get -qq update
!apt-get -qq install -y git

Created MCP workspace directory: /content/mcp_workspace
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## Step 6: Connect to MCP servers from your agent runtime

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(WORKDIR)],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", str(WORKDIR)],
    },
}

client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
print("MCP client initialized with connections for filesystem and git servers.")

MCP client initialized with connections for filesystem and git servers.


## Step 7: Build a Gemini agent that can use these tools

In [ ]:
import asyncio
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.agents.react.base import create_react_agent # Attempting import from langchain_experimental
from langchain.agents.agent import AgentExecutor
from langchain_core.prompts import PromptTemplate

# Get the tools from the MCP client
# We run this in an event loop because client.get_tools() is an async function.
# In a regular Python script, you would use await client.get_tools().
tools = asyncio.get_event_loop().run_until_complete(client.get_tools())
print(f"Retrieved {len(tools)} tools:")
for tool in tools:
    print(f"- {tool.name}: {tool.description}")

# Initialize the Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-pro")

# Define the prompt for the agent
prompt = PromptTemplate.from_template("""
You are a helpful AI assistant. You have access to the following tools:
{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

# Create the ReAct agent
agent = create_react_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Gemini agent created and ready to use.")

ModuleNotFoundError: No module named 'langchain_experimental'

In [ ]:
%pip install -qU langchain-experimental

In [ ]:
import asyncio
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.agents.react.base import create_react_agent
from langchain.agents.agent import AgentExecutor
from langchain_core.prompts import PromptTemplate

# Get the tools from the MCP client
# We run this in an event loop because client.get_tools() is an async function.
# In a regular Python script, you would use await client.get_tools().
tools = asyncio.get_event_loop().run_until_complete(client.get_tools())
print(f"Retrieved {len(tools)} tools:")
for tool in tools:
    print(f"- {tool.name}: {tool.description}")

# Initialize the Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-pro")

# Define the prompt for the agent
prompt = PromptTemplate.from_template("""
You are a helpful AI assistant. You have access to the following tools:
{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

# Create the ReAct agent
agent = create_react_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Gemini agent created and ready to use.")

### Test the Gemini Agent

Let's give our agent a simple task to see if it can interact with the MCP servers through the tools provided.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# Run the agent with a sample query
# For example, ask it to create a file using the filesystem server
response = await agent_executor.ainvoke({"input": "Create a file named 'test_agent.txt' in the workspace directory with the content 'Hello from Gemini agent!'"})

print(response["output"])

NameError: name 'agent_executor' is not defined

In [ ]:
from google.colab import userdata
userdata.get('secretName')

In [ ]:
# Run the agent with a sample query
# For example, ask it to create a file using the filesystem server
response = await agent_executor.ainvoke({"input": "Create a file named 'test_agent.txt' in the workspace directory with the content 'Hello from Gemini agent!'"})

print(response["output"])